# 📘 Annotated edition · 01_profile

A copy of `notebooks/01_profile.ipynb`. Original cells and their saved outputs are unchanged. **📘 Explanation** cells explain each cell line by line and how to read its output. **✅ Answer** cells answer every question the notebook asks: the interview angles and the "Before you move on" list.

**The point of this notebook in one sentence:** before joining or counting anything, find out what each file *really* contains (encoding, date format, grain, id format, null rates, units), because every later number depends on it.

**Key facts this notebook establishes (from the saved run):**
- 368 files, 49.9 MB, in 11 families.
- Legacy bookings are **Latin-1**, with **day-first dates** and occasionally a **repeated header line** (21 across the pack, found in notebook 02).
- Both booking feeds are at **booking-test grain**, about 2.35 rows per booking. `len()` counts tests, not bookings.
- Invoice `amount` is **text** (`"1,499.00"`). Payment amounts are in **paise** (1 rupee = 100 paise).
- 46 legacy rows (19 distinct ids) in the biggest file have a **7-digit suffix ending in 0**.
- Only 70.5% of payment references match the invoice id format exactly. After normalisation, 100% do.

> ℹ️ **One line changed in this annotated copy:** `ROOT = ... if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else ...`. The original only recognised a folder called `notebooks`. Run from `notebook_new/`, it would have looked for `notebook_new/data/raw` and failed. Nothing else in the code was changed.

# 01 · Profile every file before you trust any of it

**Produces:** `data/clean/profile_files.csv` (one row per file) and `data/clean/profile_columns.csv` (one row per column per table family), plus the first entries of the decisions log.

**Needs:** the data pack in `data/raw/`.

The rule this notebook enforces: a count is only as true as the step that recorded it. Before any join, you know every file's shape, encoding, date format, id format, null rate and date range, and you have written down what surprised you.

#### 📘 Explanation: the header

- **Produces:** two profiling tables. `profile_files.csv` has one row per file (path, family, size). `profile_columns.csv` has one row per column per table family (dtype, null rate, distinct count, min, max).
- **"A count is only as true as the step that recorded it":** a row count from a misread file (wrong encoding, header lines counted as data, wrong grain) is wrong however carefully you analyse it later. Profiling is how you earn the right to trust the counts.
- The checklist "shape, encoding, date format, id format, null rate, date range" is what each section below works through.

## CONFIG

#### 📘 Explanation: the CONFIG cell, line by line

This cell is the same in notebooks 01–06. Explained once in full:

| Code | What it does and why |
|---|---|
| `import sys, pathlib, time` | `sys` to edit the import path, `pathlib` for file paths that work on Windows and Linux, `time` to time steps. |
| `ROOT = pathlib.Path.cwd().parent if ... == "notebooks" else pathlib.Path.cwd()` | Finds the **project root**. Jupyter usually starts inside the notebook's folder, so the root is one level up. When run from the root itself (for example a script), `cwd()` is already the root. Every path below is built from `ROOT`, so the notebook works on any machine. |
| `sys.path.insert(0, str(ROOT / "src"))` | Puts `src/` at the front of Python's import search path, so `from kalpa import ...` finds the project's own package without `pip install`. |
| `import numpy as np` / `import pandas as pd` | The two core libraries: arrays and fast maths / tables. |
| `from kalpa import loaders, models, checks, logs` | The shared project package: **loaders** read the raw files, **models** are pydantic "contracts" for a valid record, **checks** print PASS/FAIL, **logs** append rows to the markdown logs in `docs/`. |
| `pd.set_option("display.max_columns", 40)` / `("display.width", 160)` | Print wide tables in full instead of truncating them with `...`. |
| `CONFIG = {...}` | All run-specific settings in one place (see notebook 00): brief, group, `raw` and `clean` folders as `pathlib` paths, the two half-years being compared, the seed, and `workers` (`None` means use every CPU core in the process pool). |
| `CONFIG["clean"].mkdir(parents=True, exist_ok=True)` | Creates `data/clean/` if it doesn't exist, and doesn't fail if it does. A fresh clone can run straight away. |
| `rng = np.random.default_rng(CONFIG["seed"])` | A seeded random generator **object**. Anything random (the shuffle test in 05) uses `rng`, so results are reproducible. |
| `print(pd.__version__, np.__version__)` | Prints library versions (**2.3.3** and **2.3.5** in the saved run). If a re-run elsewhere gives different numbers, a version difference is the first suspect. pandas 3.x, for example, changed the default string dtype. |

In [1]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "<LETTER>",
    "group": "<N>",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

2.3.3 2.3.5


## 1 · MAP: what is in the pack

The pack is an export of six source systems. Each family has its own shape. The first move is to list every file with its size and family, so you know what you are dealing with before you read a byte of it.

#### 📘 Explanation: MAP

The pack is an export of **six source systems**: legacy booking system, new "nova" booking system, the phlebotomy (collections) app, the lab (results), billing (invoices) and the payment gateway. There's also a patient master, campaigns, reference tables and the plan. Each system has its own format. Listing every file first tells you the size and shape of the job before you read any content.

#### 📘 Explanation: listing the pack

- `loaders.list_pack(CONFIG["raw"])` walks every file under `data/raw` (`rglob("*")`) and returns a DataFrame with `path`, `family`, `bytes`, `ext`. Family is `legacy` / `nova` / `results` when that word is in the path, otherwise the parent folder name (`invoices`, `payments`, ...).
- `files['bytes'].sum()/1e6`: total size in megabytes.
- `files.groupby("family").agg(files=("path","count"), mb=("bytes", lambda b: round(b.sum()/1e6, 1)))` uses **named aggregation**: for each family, count the files and sum the bytes into MB. `.sort_values("mb", ascending=False)` puts the biggest first.

In [2]:
files = loaders.list_pack(CONFIG["raw"])
print(f"{len(files)} files, {files['bytes'].sum()/1e6:.1f} MB")
files.groupby("family").agg(files=("path", "count"), mb=("bytes", lambda b: round(b.sum()/1e6, 1))).sort_values("mb", ascending=False)

368 files, 49.9 MB


,files,mb
family,,
legacy,102,15.2
results,108,10.5
invoices,7,6.5
patients,1,6.3
payments,28,6.2
collections,108,4.4
nova,6,0.7
campaigns,2,0.1
raw,2,0.1


#### 📘 Reading the output

- **368 files, 49.9 MB**, the whole pack.
- `legacy` has 102 files (6 cities × 17–18 months; Chennai and Jaipur stop after March 2026 when they moved to nova). `results` and `collections` have 108 each (6 cities × 18 months). `nova` has only 6 files (2 cities × 3 months, April–June 2026).
- `raw` with 2 files is the README and MANIFEST at the top of `data/raw` (their parent folder is called `raw`).
- `patients` is a single 6.3 MB file, the full patient master.

**What to take from it:** the bookings data is split between two systems by **city and time**. Chennai and Jaipur switch system in April 2026. Any comparison across that date compares two different recording systems, which is brief B's problem in one line.

### CHECK

#### 📘 Explanation: verifying the download against MANIFEST.json

- `json.load(open(... "MANIFEST.json"))` reads the manifest, which lists every file with its size and **SHA-256** hash (a fingerprint: change one byte and the hash changes completely).
- `listed = {f["path"]: f for f in manifest["files"]}`: a dictionary from path to manifest entry.
- `missing = [p for p in listed if not (CONFIG["raw"] / p).exists()]`: files the manifest promises but the disk doesn't have.
- `checks.check("every file in MANIFEST.json is present", len(missing) == 0, ...)` prints PASS/FAIL.
- `sample = list(listed)[:: max(1, len(listed)//3)][:3]` takes every n-th file to get **3 spread-out files**. Hashing all of them is slow and unnecessary for a sanity check: a broken download usually breaks many files.
- `hashlib.sha256(...read_bytes()).hexdigest() == listed[p]["sha256"]` recomputes each sampled file's hash and compares it with the manifest.

In [3]:
import json, hashlib
manifest = json.load(open(CONFIG["raw"] / "MANIFEST.json"))
listed = {f["path"]: f for f in manifest["files"]}
missing = [p for p in listed if not (CONFIG["raw"] / p).exists()]
checks.check("every file in MANIFEST.json is present", len(missing) == 0, f"{len(missing)} missing")
# spot-check three hashes rather than all 300: enough to know the download is intact
sample = list(listed)[:: max(1, len(listed)//3)][:3]
ok = all(hashlib.sha256((CONFIG["raw"] / p).read_bytes()).hexdigest() == listed[p]["sha256"] for p in sample)
checks.check("spot-checked SHA-256 match the manifest", ok, sample)

PASS  every file in MANIFEST.json is present   0 missing
PASS  spot-checked SHA-256 match the manifest   ['README.md', 'collections/collections_AHM_2025-12.csv.gz', 'payments/gateway_export_2026-05.csv.gz']


True

#### 📘 Reading the output

Both checks **PASS**: 0 files missing, and the 3 sampled hashes (README, a collections file, a payments file) match. The data on disk is exactly what was released. The trailing `True` is just the return value of the last `check()` call, which Jupyter displays.

## 2 · DO: read one legacy file the naive way, and watch it go wrong

A legacy monthly export is Latin-1, dates are day first, and the vendor appended files so some carry the header twice. Read one naively first. The failure is the lesson.

#### 📘 Explanation: why read it wrong first

The legacy vendor's files have three traps: **Latin-1 encoding** (not UTF-8), **day-first dates** (`13/02/2026` = 13 February), and a **header line repeated mid-file** where monthly files were appended together. Seeing the naive read fail teaches you to spot the symptom when it happens with other data.

#### 📘 Explanation: the naive read

- `sorted((... / "legacy").rglob("*.csv.gz"))`: every legacy file, recursively, in a stable order.
- `biggest = lambda paths: max(paths, key=lambda p: p.stat().st_size)`: picks the largest file. The comment explains why: the **largest file is the most representative sample**. The smallest is often an edge case such as a partial month.
- `pd.read_csv(one)` uses the default encoding **UTF-8**.
- `try/except UnicodeDecodeError`: catches the failure and prints it instead of crashing.

In [4]:
legacy_files = sorted((CONFIG["raw"] / "bookings" / "legacy").rglob("*.csv.gz"))
biggest = lambda paths: max(paths, key=lambda p: p.stat().st_size)   # the largest file of a family is the honest sample; the smallest is often an edge
one = biggest(legacy_files)
print(one.name)
try:
    naive = pd.read_csv(one)           # default encoding is UTF-8
    print("read without error:", naive.shape)
except UnicodeDecodeError as err:
    print("UnicodeDecodeError:", str(err)[:120])

bookings_2026-02.csv.gz
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe9 in position 3541: invalid continuation byte


#### 📘 Reading the output

`bookings_2026-02.csv.gz` (Bengaluru, Feb 2026) is the biggest file. The read fails with `'utf-8' codec can't decode byte 0xe9 in position 3541`. Byte `0xE9` is **é** in Latin-1, for example in an accented patient name. In UTF-8, é is two bytes (`0xC3 0xA9`), and a lone `0xE9` is invalid. So the file is not UTF-8.

#### 📘 Explanation: reading with the right encoding, still naively

- `encoding="latin-1"`: Latin-1 maps each of the 256 byte values to one character, so it **never** raises. That's also why it's dangerous as a guess: a UTF-8 file read as Latin-1 "works" and silently turns `é` into `Ã©`.
- `print(naive.dtypes)`: shows what pandas guessed for each column.
- `(naive["booking_id"] == "booking_id").sum()` counts rows whose id is literally the text "booking_id", which means a repeated header line read as data.
- `type(naive["slot_at"].iloc[0]).__name__`: what Python type the first `slot_at` value is.

In [5]:
naive = pd.read_csv(one, encoding="latin-1")
print(naive.dtypes)
print()
print("rows whose booking_id is literally the header:", (naive["booking_id"] == "booking_id").sum())
print("slot_at parsed as:", type(naive["slot_at"].iloc[0]).__name__, "->", naive["slot_at"].iloc[0])

booking_id      object
created_at      object
slot_at         object
city            object
clinic_id       object
channel         object
source          object
patient_id      object
patient_name    object
test_code       object
test_price       int64
home_fee         int64
dtype: object

rows whose booking_id is literally the header: 0
slot_at parsed as: str -> 13/02/2026 09:45


#### 📘 Reading the output

- Almost every column is `object` (Python strings). Only `test_price` and `home_fee` became `int64`. **Dates are not dates yet:** `slot_at` is the string `"13/02/2026 09:45"`.
- 0 repeated header lines in *this* file. Notebook 02 finds **21** across the whole pack. Checking one file is not enough.
- If you let pandas guess date formats, `"03/02/2026"` could be read as **3 March** (month-first, US style) instead of 3 February. A value like `13/02` can't be month-first, so a guesser may switch format row by row and produce a mixture. The fix is an explicit `format="%d/%m/%Y %H:%M"`.

> **Interview angle.** Why did `pd.read_csv` fail on the first attempt, and why would guessing the encoding be a mistake?
>
> *What a good answer covers:* The file is Latin-1; UTF-8 is the default. Guessing by trying encodings until one works can silently mangle accented characters into different, valid characters. Find the encoding from the source system or the vendor and state it in the loader.

#### ✅ Answer: why did `pd.read_csv` fail, and why is guessing the encoding a mistake?

**Why it failed:** `read_csv` decodes bytes as UTF-8 by default. The legacy vendor writes **Latin-1** (ISO-8859-1), where `é` is the single byte `0xE9`. In UTF-8 that byte must start a multi-byte sequence, and the next byte doesn't continue one, so decoding stops with `UnicodeDecodeError` at position 3541.

**Why guessing is a mistake:** you'd try encodings until one doesn't raise, but *not raising* doesn't mean *correct*:
- **Latin-1 never raises**: every byte maps to some character. So "try UTF-8, then Latin-1" always "succeeds", even on a file that is actually Windows-1252 or UTF-8. A UTF-8 file read as Latin-1 turns `Désai` into `DÃ©sai`, valid but wrong text. That breaks name matching (patient de-duplication in notebook 03) without any error.
- Windows-1252 and Latin-1 differ in bytes 0x80–0x9F (curly quotes, the euro sign, dashes), so the "wrong but working" encoding can corrupt a few characters in a way you'd never notice.
- The right answer depends on the **source system**, not on your file. The next file from the same vendor might contain bytes your guess happened not to meet.

**What to do instead:** find out the encoding (the data README says "Encoding and date format are the vendor's", so ask the data team or vendor), **state it explicitly in the loader** (`loaders.read_legacy_file` uses `encoding="latin-1"`), and record it in the decisions log so it's auditable.

## 3 · DO: the proper reader

`loaders.read_legacy_file` reads with the right encoding, drops repeated header lines and counts them, parses the day-first dates with an explicit format, strips the ids, and declares dtypes so a 200,000-row file costs a tenth of the memory it would as plain objects.

#### 📘 Explanation: what the proper reader does (`loaders.read_legacy_file`)

Step by step, from `src/kalpa/loaders.py`:
1. `gzip.open(path, "rt", encoding="latin-1")`: decompress and decode with the **stated** encoding.
2. `lines = text.splitlines(); header = lines[0]; body = [ln for ln in lines[1:] if ln != header]` drops every line identical to the header. These are append artefacts.
3. `repeated = len(lines) - 1 - len(body)` counts them, stored in `df.attrs["repeated_header_lines"]` so the count can go in the log.
4. `pd.read_csv(io.StringIO(...), dtype=LEGACY_DTYPES)` parses with **declared dtypes**: ids as `string`, low-cardinality text (city, clinic, channel, source, test_code) as `category`, prices as nullable `Int32`.
5. `pd.to_datetime(df[col], format="%d/%m/%Y %H:%M", errors="coerce")` parses dates with an **explicit day-first format**. Anything unparsable becomes `NaT` instead of raising, so it can be counted.
6. `df["booking_id"].str.strip()` removes leading and trailing spaces in ids. These were found in the raw file: `'BK-BLR-202602-092170 '`.
7. Adds `source_file` (lineage: which file each row came from) and `system="legacy"`.

**Why read the text first instead of `pd.read_csv(..., encoding=...)` directly?** Because the repeated header lines have to be removed *before* parsing. Otherwise pandas reads `"test_price"` as a value in an integer column and fails, or turns the whole column into text.

#### 📘 Explanation: timing and memory of the proper read

- `t0 = time.time()` ... `time.time()-t0`: wall-clock time of the read.
- `df.attrs['repeated_header_lines']`: the count the loader stored.
- `df.memory_usage(deep=True).sum()/1e6`: **real** memory in MB. `deep=True` measures the actual string contents, not just pointer sizes.
- `df.head(3)`: the first three parsed rows.

In [6]:
t0 = time.time()
df = loaders.read_legacy_file(one)
print(f"{len(df):,} booking-test rows in {time.time()-t0:.2f}s; repeated header lines dropped: {df.attrs['repeated_header_lines']}")
print(df.dtypes)
print(f"memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB  (naive object frame: {naive.memory_usage(deep=True).sum()/1e6:.1f} MB)")
df.head(3)

19,602 booking-test rows in 0.47s; repeated header lines dropped: 0
booking_id      string[python]
created_at      datetime64[ns]
slot_at         datetime64[ns]
city                  category
clinic_id             category
channel               category
source                category
patient_id      string[python]
patient_name    string[python]
test_code             category
test_price               Int32
home_fee                 Int32
source_file             object
system                  object
dtype: object
memory: 6.8 MB  (naive object frame: 11.9 MB)


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,system
0,BK-BLR-202602-092121,2026-02-13 05:39:00,2026-02-13 09:45:00,Bengaluru,BLR-01,home,app,PBLR0024184,Sneha Fernandes,PK04,2499,0,bookings_2026-02.csv.gz,legacy
1,BK-BLR-202602-092122,2026-02-25 04:57:00,2026-02-25 13:41:00,Bengaluru,BLR-01,clinic,app,PBLR0060190,Nisha Menon,T016,160,0,bookings_2026-02.csv.gz,legacy
2,BK-BLR-202602-092123,2026-01-31 04:21:00,2026-02-01 10:16:00,Bengaluru,BLR-03,home,app,PBLR0054249,Aarav Singh,T007,550,0,bookings_2026-02.csv.gz,legacy


#### 📘 Reading the output

- **19,602 booking-test rows in 0.47 s**, with 0 repeated headers in this file.
- dtypes are now meaningful: `datetime64[ns]` for both dates, `category` for city, clinic, channel, source and test_code, `string` for ids and names, nullable `Int32` for prices.
- **Memory: 6.8 MB against 11.9 MB** for the naive frame, about 43% less. Over 1.25 million legacy rows that's the difference between ~430 MB and ~760 MB of RAM.
- In `head(3)`, `BK-BLR-202602-092122` appears once per **test**. One booking with 3 tests is 3 rows. `created_at` 2026-02-13 05:39 and `slot_at` 2026-02-13 09:45 are both real timestamps now.
- Why `Int32` (capital I) and not `int32`? The capital version is pandas' **nullable** integer, which can hold `<NA>`. A plain `int` column can't hold missing values and would be converted to `float64`.

> **Interview angle.** What does declaring `dtype={'city': 'category'}` change, and when is `category` the wrong choice?
>
> *What a good answer covers:* A category stores each distinct value once and an integer code per row, so memory drops and groupby gets faster. It is wrong for high-cardinality columns such as ids, where the codes table is as large as the data, and it needs care on concat when the two frames have different category sets.

#### ✅ Answer: what does `dtype={'city': 'category'}` change, and when is `category` the wrong choice?

**What it changes:** a `category` column stores each **distinct value once** (the "categories") and each row as a small integer code pointing to it. `city` has 6 distinct values across 1.3 million rows, so instead of 1.3 million Python string objects (~60 bytes each) you store 1.3 million `int8` codes (1 byte each) plus 6 strings.
- **Memory** drops sharply (part of the 11.9 → 6.8 MB saving above).
- **groupby / value_counts get faster**, because pandas groups on integer codes instead of comparing strings.
- **Sorting** can follow a meaningful order if the categories are ordered (for example `morning < midday < evening` in notebook 05).

**When it's wrong:**
1. **High-cardinality columns** such as `booking_id` or `patient_id`, where almost every value is unique. The categories table is as big as the data, so nothing is saved, and you pay the overhead of codes on top.
2. **Concatenating frames with different category sets.** If file A's `test_code` categories are {T001, T002} and file B's are {T002, PK04}, `pd.concat` silently **falls back to `object`**, and the memory saving is lost. Notebook 02's `all_rows["city"].astype("string")` is partly a response to this. Use `union_categoricals` or set the full category list explicitly.
3. **Columns you'll modify with new values.** Assigning a value that isn't an existing category raises an error (`Cannot setitem on a Categorical with a new category`).
4. **groupby with `observed=False`** (the old default) produces rows for *every* category combination, even empty ones. That's why later notebooks pass `observed=True`.

## 4 · DO: profile every family with the same function

One function, applied to every table family: rows, distinct keys, null rate per column, min and max date, and the share of ids matching the expected format. Run it on one file per family here; notebook 02 runs the full load.

#### 📘 Explanation: one profile function for every family

The idea is **consistency**: if each table is profiled with ad-hoc code, you'll check null rates on one and forget them on another. One function applied to every family gives directly comparable outputs. It's run on **one file per family** (the biggest). The full load is notebook 02's job, which keeps this notebook fast.

#### 📘 Explanation: the `profile` function and the per-family calls

```python
def profile(df, name, key=None):
    for col in df.columns:
        s = df[col]
        rows.append({... "dtype": str(s.dtype),
                     "null_rate": round(float(s.isna().mean()), 4),   # share of missing values
                     "distinct": int(s.nunique(dropna=True)),          # number of distinct non-null values
                     "min"/"max": only for numeric or datetime columns (text min/max is meaningless) })
```
- `s.isna().mean()`: `isna()` gives True/False per row. The mean of booleans is the **share** of True, so this is the null rate.
- If `key` is given, it prints **rows per key** = `len(df) / df[key].nunique()`. This is the **grain check**: 1.00 means one row per key, more than 1 means the table is at a finer grain or has duplicates.

Then it's applied to the biggest file of each family:
- legacy bookings (`df`, already loaded), and nova (`loaders.read_nova_file`, which flattens the nested JSON tests into one row per test).
- collections (`parse_dates=["collected_at"]`), results, invoices (with `dtype={"amount": "string"}` so amounts aren't guessed), payments (`gateway_*` files only, not settlement files), and patients.
- `pd.concat(prof, ignore_index=True)` stacks all the per-table profiles into one long table.

In [7]:
def profile(df: pd.DataFrame, name: str, key: str | None = None) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({"table": name, "column": col, "dtype": str(s.dtype), "null_rate": round(float(s.isna().mean()), 4),
                     "distinct": int(s.nunique(dropna=True)),
                     "min": s.min() if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) else "",
                     "max": s.max() if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) else ""})
    out = pd.DataFrame(rows)
    if key is not None:
        print(f"{name}: {len(df):,} rows, {df[key].nunique():,} distinct {key}  ->  {len(df)/max(1, df[key].nunique()):.2f} rows per key")
    return out

prof = [profile(df, "bookings_legacy", "booking_id")]
nova_files = sorted((CONFIG["raw"] / "bookings" / "nova").rglob("*.jsonl.gz"))
if nova_files:
    nv = loaders.read_nova_file(biggest(nova_files))
    prof.append(profile(nv, "bookings_nova", "booking_id"))
coll = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "collections").glob("*.csv.gz"))), parse_dates=["collected_at"])
prof.append(profile(coll, "collections", "booking_id"))
res = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "results").rglob("*.csv.gz"))))
prof.append(profile(res, "results", "booking_id"))
inv = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "invoices").glob("*.csv.gz"))), dtype={"amount": "string"})
prof.append(profile(inv, "invoices", "invoice_id"))
pay = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "payments").glob("gateway_*.csv.gz"))))
prof.append(profile(pay, "payments", "payment_id"))
pat = pd.read_csv(CONFIG["raw"] / "patients" / "patients.csv.gz")
prof.append(profile(pat, "patients", "patient_id"))
profile_columns = pd.concat(prof, ignore_index=True)
profile_columns

bookings_legacy: 19,602 rows, 8,344 distinct booking_id  ->  2.35 rows per key


bookings_nova: 13,966 rows, 5,903 distinct booking_id  ->  2.37 rows per key
collections: 8,161 rows, 8,120 distinct booking_id  ->  1.01 rows per key
results: 16,310 rows, 7,084 distinct booking_id  ->  2.30 rows per key
invoices: 83,630 rows, 83,630 distinct invoice_id  ->  1.00 rows per key
payments: 28,758 rows, 28,758 distinct payment_id  ->  1.00 rows per key
patients: 309,217 rows, 309,217 distinct patient_id  ->  1.00 rows per key


,table,column,dtype,null_rate,distinct,min,max
0,bookings_legacy,booking_id,string,0.0,8344,,
1,bookings_legacy,created_at,datetime64[ns],0.0,7414,2026-01-26 20:51:00,2026-03-01 17:56:00
2,bookings_legacy,slot_at,datetime64[ns],0.0,6751,2026-02-01 06:00:00,2026-02-28 19:56:00
3,bookings_legacy,city,category,0.0,1,,
4,bookings_legacy,clinic_id,category,0.0,6,,
...,...,...,...,...,...,...,...
60,patients,patient_name,object,0.0,1200,,
61,patients,phone,object,0.0,309200,,
62,patients,dob,object,0.0,24456,,
63,patients,city,object,0.0,6,,


#### 📘 Reading the output: the grain line is the most important result

| Table | Rows | Distinct key | Rows per key | Meaning |
|---|---|---|---|---|
| bookings_legacy | 19,602 | 8,344 booking_id | **2.35** | one row per **test** within a booking |
| bookings_nova | 13,966 | 5,903 | **2.37** | same grain after flattening |
| collections | 8,161 | 8,120 | **1.01** | should be 1. The extra 1% are **double scans** (removed in 03) |
| results | 16,310 | 7,084 | 2.30 | one row per test run |
| invoices | 83,630 | 83,630 | 1.00 | invoice_id is unique |
| payments | 28,758 | 28,758 | 1.00 | payment_id unique (but several payments can point to one invoice) |
| patients | 309,217 | 309,217 | 1.00 | patient_id unique, though one *person* may have several ids (see 03) |

In the column table you also see, for example, `city` with **1 distinct value**: this file is one city. `created_at` runs from 26 Jan, before February, because people book days ahead of their slot.

### SEE: the things a profile shows that a head() does not

- `rows per key` above 1.0 on a bookings file means the grain is booking-test, not booking. Count bookings with `nunique`, never with `len`.
- An `amount` column that profiles as `string` holds thousands separators. Sum it and pandas raises, or worse, concatenates.
- A `created_at_epoch` column that profiles as integer with a min near 1.7 billion is Unix seconds.
- A null rate of 0.003 on `test_code` is small until you remember that every one of those rows is a test you cannot price.

#### 📘 Explanation: SEE, what a profile shows that `head()` doesn't

1. **Rows per key above 1.0 means booking-test grain.** `len(bookings)` is a count of *tests*. Count bookings with `nunique()`, or first roll up to one row per booking (notebook 02 does this).
2. **`amount` profiled as `string`** means it contains thousands separators (`"1,499.00"`). Summing strings in pandas *concatenates* them or raises. Parse it with `.str.replace(",", "").astype(float)` (done in 03).
3. **`created_at_epoch` as an integer near 1.7 billion** means Unix seconds since 1970-01-01 UTC. 1.7 billion seconds is about 54 years, so 2024–2026. Convert with `pd.to_datetime(..., unit="s")`, then shift UTC to IST (+5:30), as done in 03.
4. **A 0.3% null rate on `test_code`** sounds negligible, but each such row is a test you **can't price or classify**. Over 1.3 million rows that's thousands of tests. The validator in 02 rejects bookings whose test code is empty: 1,287 of the 3,107 rejects.

#### 📘 Explanation: four checks on the profile

1. `df["booking_id"].str.match(loaders.ID_PATTERNS["legacy"].pattern)` tests each id against `^BK-[A-Z]{3}-\d{6}-\d{6}$`: `BK-`, 3 capital letters (city), 6 digits (YYYYMM), 6 digits (sequence). `.all()` is True only if every row matches. `(~id_ok).sum()` counts the failures.
2. `df["slot_at"].notna().all()`: every date parsed (no `NaT` from `errors="coerce"`).
3. `pd.api.types.is_string_dtype(inv["amount"])` confirms amount is text, so we know to parse it.
4. Payments: amount is an integer and its median is above 1,000, which is consistent with **paise**. A median test price of ~₹1,250 is 125,000 paise.

In [8]:
id_ok = df["booking_id"].str.match(loaders.ID_PATTERNS["legacy"].pattern)
checks.check("every legacy booking id matches BK-CCC-YYYYMM-NNNNNN after strip", bool(id_ok.all()), f"{(~id_ok).sum()} do not")
checks.check("legacy slot_at parsed for every row", bool(df["slot_at"].notna().all()), f"{df['slot_at'].isna().sum()} NaT")
checks.check("invoice amount is text with separators (so we know to parse it)", pd.api.types.is_string_dtype(inv["amount"]), inv["amount"].iloc[0])
checks.check("payments amount is in paise (integer, large)", pd.api.types.is_integer_dtype(pay["amount_paise"]) and pay["amount_paise"].median() > 1000, pay["amount_paise"].median())

FAIL  every legacy booking id matches BK-CCC-YYYYMM-NNNNNN after strip   46 do not
PASS  legacy slot_at parsed for every row   0 NaT
PASS  invoice amount is text with separators (so we know to parse it)   1,499.00
PASS  payments amount is in paise (integer, large)   125000.0


True

#### 📘 Reading the output: one FAIL, and why that's fine

- **FAIL, 46 legacy ids don't match after strip.** These 46 rows are **19 distinct bookings**, and their ids have **7 digits** at the end instead of 6, for example `BK-BLR-202602-0921870`. Across the whole pack the validator in notebook 02 rejects **793 bookings** for this, and **every one of them ends in an extra `0`**. When the trailing 0 is removed, none of the trimmed ids collides with an existing booking. So these look like real bookings whose id the vendor corrupted by appending a digit, not duplicates. Repairing them ("drop a trailing 0 from 7-digit suffixes") would be a reversible, logged decision. The notebook currently rejects them.
- The check is designed to **print FAIL and continue**. The failure is information, not a crash.
- **PASS** for dates, the text amount (`"1,499.00"`) and paise (median 125,000 paise = ₹1,250).

## 5 · DO: the identifier audit

Every table that should join on an id gets its id format checked against a pattern. Mismatches are not errors yet; they are a list of things to decide about in notebook 03.

#### 📘 Explanation: the identifier audit

Every join in this project is on an id: booking ↔ collection ↔ result on `booking_id`, invoice ↔ payment on `invoice_id`. If an id is formatted differently on the two sides, the join silently **drops** rows (inner join) or leaves them **unmatched** (left join). The audit measures, per table, what share of ids follow the expected pattern. A mismatch is not yet an error. It's a decision to make in notebook 03.

#### 📘 Explanation: the audit dictionary

Each entry is `series.str.match(regex).mean()`, the **share** of rows matching:
- legacy `booking_id` **before** strip, from the naive frame, so trailing spaces still count as failures.
- collections and results `booking_id`, where the pattern allows **either** system's format: `BK-XXX-YYYYMM-NNNNNN` or `nova_xxx_NNNNNNN` (the `|` in the regex means OR).
- `invoice_id` against `INV-YYYY-NNNNNN`.
- payment `notes_invoice_ref` **as typed** against the same pattern, and **after** `loaders.normalise_invoice_ref`, which upper-cases, strips spaces, removes hyphens, then re-inserts them in the canonical places: `inv-2026-000123 ` and `INV2026000123` both become `INV-2026-000123`.

In [9]:
audit = {
    "legacy booking_id (raw, before strip)": naive["booking_id"].astype("string").str.match(r"^BK-[A-Z]{3}-\d{6}-\d{6}$").mean(),
    "collections booking_id": coll["booking_id"].astype("string").str.match(r"^(BK-[A-Z]{3}-\d{6}-\d{6}|nova_[a-z]{3}_\d{7})$").mean(),
    "results booking_id": res["booking_id"].astype("string").str.match(r"^(BK-[A-Z]{3}-\d{6}-\d{6}|nova_[a-z]{3}_\d{7})$").mean(),
    "invoice_id": inv["invoice_id"].astype("string").str.match(r"^INV-\d{4}-\d{6}$").mean(),
    "payment notes_invoice_ref, exact format": pay["notes_invoice_ref"].astype("string").str.match(r"^INV-\d{4}-\d{6}$").mean(),
    "payment notes_invoice_ref, after normalisation": loaders.normalise_invoice_ref(pay["notes_invoice_ref"]).str.match(r"^INV-\d{4}-\d{6}$").mean(),
}
pd.Series(audit, name="share_matching_pattern").round(4).to_frame()

,share_matching_pattern
"legacy booking_id (raw, before strip)",0.9877
collections booking_id,0.9978
results booking_id,0.9977
invoice_id,1.0000
"payment notes_invoice_ref, exact format",0.7051
"payment notes_invoice_ref, after normalisation",1.0000


#### 📘 Reading the output

| Id | Share matching | Interpretation |
|---|---|---|
| legacy booking_id (raw) | 98.77% | 1.23% fail: trailing spaces (fixed by strip) plus the 7-digit ids |
| collections / results booking_id | 99.78% / 99.77% | a few malformed ids that won't join back to bookings |
| invoice_id | 100% | billing's ids are clean |
| payment ref, exact | **70.51%** | almost 30% of payment references are typed in another format |
| payment ref, normalised | **100%** | all of them become valid after the normalisation rule |

Without normalisation, ~30% of payments would look unpaid or orphaned. That's a fake "leak" created by formatting. Note that *matching the pattern* doesn't mean *matching an invoice*. Notebook 03 finds 4,588 captured payments whose well-formed reference points to an invoice that doesn't exist.

> **Interview angle.** A payment reference is 'inv-2026-000123 ' with a trailing space. Is normalising it a cleaning decision or a parsing decision, and who should be able to audit it?
>
> *What a good answer covers:* It is a decision: you are asserting that this string identifies that invoice. It is reversible and low-risk, so it belongs in the decisions log with its rule and the count of rows it changed. A transposed digit is different: it cannot be repaired without guessing, so it stays unmatched and is listed.

#### ✅ Answer: a payment reference is `'inv-2026-000123 '`. Is normalising it a cleaning decision or a parsing decision, and who should be able to audit it?

**It's a cleaning decision, even though it looks like parsing.** Parsing turns text into the type it already unambiguously represents (`"1,499.00"` → 1499.0). Here you're **asserting an identity**: that the free text a customer or gateway typed *refers to* invoice `INV-2026-000123`. That's a business claim, and it could be wrong.

**Why it's an acceptable decision:**
- It's **low-risk**: changing case, trimming whitespace and moving hyphens can't turn one valid invoice number into a *different* valid one. The digits are untouched.
- It's **reversible**: the original `notes_invoice_ref` column is kept beside the normalised `invoice_id`.
- It's **high-impact**: 29.5% of payment rows depend on it.

**So it goes in `docs/decisions-log.md`** with the rule, the number of rows it changed (29.5% of the rows in the profiled month. Recount it on all 473,507 payment rows when you log it), and its effect on the headline: unmatched payments fall from ~30% to the residual orphans.

**Who audits it:** the **finance controller** (it decides which money counts as collected), and any analyst re-running the reconciliation. The rule must therefore live in shared, tested code (`loaders.normalise_invoice_ref`, with tests in `tests/`), not in a one-off notebook cell.

**Where the line is:** a **transposed digit** (`INV-2026-000132` typed for `...000123`) can't be repaired without *guessing*, and a guess could credit the wrong customer. It stays unmatched and goes on a list for finance. The matching fallback (amount and date) is a separate, explicitly logged step.

## 6 · SUM: write the profile out and log the first decisions

#### 📘 Explanation: writing outputs and logging a decision

- `files.to_csv(...)` and `profile_columns.to_csv(...)` save both profile tables to `data/clean/`. `index=False` keeps the pandas index out of the file.
- `logs.decision(...)` **appends a row to the Decisions table in `docs/decisions-log.md`**: table, what was found, decision, reason, rows affected, effect on the headline, owner, number. Logging from code means the decision is recorded *at the moment it's made*, with the real count.
- `checks.summary()` prints "X of Y checks passed" and returns the table of every check run in this notebook.

⚠️ **Two problems to know about:**
1. `logs.decision` **appends on every run**. `docs/decisions-log.md` already contains this same row three times because the notebook was run three times. Either run the logging cells once, or make them idempotent (check whether the row exists before appending).
2. `rows_affected=df.attrs["repeated_header_lines"]` is **0**, because `df` is just the biggest file. Across the whole pack the count is **21** (from notebook 02's full load). The log should say 21.

In [10]:
files.to_csv(CONFIG["clean"] / "profile_files.csv", index=False)
profile_columns.to_csv(CONFIG["clean"] / "profile_columns.csv", index=False)
print("wrote profile_files.csv and profile_columns.csv")
# Example of recording a decision the moment it is made. Uncomment and fill in when you have decided.
logs.decision("bookings_legacy", "header line repeated mid-file", "drop", "an append artefact, not data",
               rows_affected=df.attrs["repeated_header_lines"], effect_on_headline="none (not rows)", owner="Rohit", n=1)
checks.summary()

wrote profile_files.csv and profile_columns.csv
5 of 6 checks passed


,check,passed,detail
0,every file in MANIFEST.json is present,True,0 missing
1,spot-checked SHA-256 match the manifest,True,"['README.md', 'collections/collections_AHM_202..."
2,every legacy booking id matches BK-CCC-YYYYMM-...,False,46 do not
3,legacy slot_at parsed for every row,True,0 NaT
4,invoice amount is text with separators (so we ...,True,"1,499.00"
5,"payments amount is in paise (integer, large)",True,125000.0


#### 📘 Reading the output

"5 of 6 checks passed". The single failure is the 46 malformed legacy ids, which is information to act on in 02 and 03. The summary table lists every check with its detail, ready to paste into the challenges log.

## Before you move on

- Which table is the source of truth for a count of tests, and why that one? Write it in `docs/approach-log.md` row 1.
- Which join will be hardest, and what did the id audit tell you about it?
- What did you expect the row counts to be before you looked? How far off were you?

#### ✅ Answers: "Before you move on"

**1. Which table is the source of truth for a count of tests, and why that one?**

It depends on which event you're counting. Name the event first:

| If the question is... | Table of record | Why |
|---|---|---|
| tests **booked / demanded** (the growth headline, comparable with the plan) | **bookings** (legacy + nova), using `n_tests` per booking after the identity rule | It's the only table with booking identity, slot time, city, channel and patient for **both years and both systems**. Its grain is booking-test (2.35 rows per booking), so count bookings with `nunique` and tests with `n_tests` or rows. |
| tests **actually run** | **results** (1,078,420 rows, 473,905 bookings) | It records the lab event itself, at test grain. 99.6% of collected bookings have results and 0% of no-shows do, so it's consistent with collections. It has 9 duplicate booking-test pairs to remove. |
| samples taken | collections | about 1 row per booking (after removing 2,894 double scans), so the wrong grain for tests |
| money | invoices / payments | invoice grain, with packages, corrections and retries. Not a test count. |

**For our headline we use bookings**, for three reasons: the plan is a *test-volume target* that covers bookings, Dr Menon's "5 percent" is reproduced exactly from bookings (+5.0%: 455,383 vs 433,611 tests, H1 2026 vs H1 2025), and only bookings carry the dimensions we need to split by. We state the alternative (results) in approach-log row 1 and say that a "tests run" headline would come from results.

**2. Which join will be hardest, and what did the id audit tell you about it?**

**Payments → invoices.** The id audit showed only **70.5%** of payment references match the invoice format exactly, so a naive join leaves about 30% of money "unmatched". After normalisation 100% are well-formed, but notebook 03 shows the remaining issues: 4,588 captured payments (₹7.27 M) still point at invoice numbers that don't exist (an `INV-2025-9xxxxx` range billing never issued), 21,820 invoices have no captured payment, and 1,925 invoices have more than one captured payment (retries or double charges). It's a many-to-one join on a free-text key, the hardest kind.

A close second, found later: **bookings → collections for rescheduled nova bookings.** The phlebotomy app logs the collection against the *original* booking id, so 1,449 rescheduled visits look uncollected (see notebook 03).

**3. What did you expect the row counts to be, and how far off were you?**

This one is personal: write your own prediction before looking. For reference, the kit README said "about 1.3 million booking-test rows across 360 files". The actual figure is **1,315,183** rows in **368** files, which is very close. What's easy to get wrong is the *grain*: if you expected 1.3 million **bookings**, you were off by a factor of ~2.3, because there are 567,986 distinct bookings. Collections (552,492 rows) came in close to the booking count, as expected for one row per visit.

#### 📘 Explanation: logging the approach

`logs.approach(...)` appends a row to `docs/approach-log.md`: the part of the work, the alternatives considered (a)–(e), the choice, why, what would change the choice, what happened, owner and number. The panel reads this log first and may cover the "chosen" column and ask you to choose again aloud. That's why the alternatives are written *before* deciding.

⚠️ Like `logs.decision`, this appends on every run. The approach log already contains this row 3 times. Also note that the log's row 1 (written by hand) chose **results**, while this cell records **bookings legacy**. Make the two agree. The ✅ answer above explains when each is right.

In [11]:
logs.approach(
    part="Which table to count tests from",
    considered="(a) Bookings legacy, count rows. (b) Bookings legacy, count distinct booking_id. (c) Collections. (d) Invoices or payments. (e) Results.",
    chosen="Bookings legacy is the source of truth for booking_id, slot_at and patient_id; other tables join to it on booking_id.",
    why="It holds the booking identity and slot time. Its grain is booking-test (about 2.35 rows per booking_id), so count bookings with nunique, never len.",
    would_change="If the question is tests actually run, results is the table of record, since bookings include no-shows and cancellations.",
    happened="Profiled: 19,602 rows for 8,344 distinct booking_id in the largest file; 46 ids fail the pattern after strip.",
    owner="Rohit",
    n=1,
)

---
### ✅ Summary of notebook 01

| Finding | Number | Where it matters |
|---|---|---|
| Pack size | 368 files, 49.9 MB, all present, hashes OK | trust in the inputs |
| Legacy encoding / dates | Latin-1, `%d/%m/%Y %H:%M` | loader in 02 |
| Grain | about 2.35 rows per booking (booking-test) | count with `nunique` / `n_tests` |
| Malformed legacy ids | 46 rows / 19 ids in the biggest file (793 bookings in the pack), extra trailing `0` | validator rejects in 02 |
| Payment refs | 70.5% exact → 100% after normalisation | invoice ↔ payment join in 03 |
| Units | invoice amount is text, payments in paise, payment time is epoch seconds UTC | parsing in 03 |